# 🏛️ Nation Optimizer: Parliamentary GRPO Training

Welcome to the central command center for the **Communism Simulator** RL pipeline. This notebook manages the entire lifecycle of our parliamentary agents—from synthesizing historical datasets to fine-tuning with **GRPO** and benchmarking the resulting model.

## 🗺️ Roadmap
1.  **Part 1**: Environment Setup & Secrets
2.  **Part 2**: Dataset Synthesis (Golden Path)
3.  **Part 3**: Interactive Reward Debugger
4.  **Part 4**: Local Smoke Test
5.  **Part 5**: Hugging Face Jobs Dispatcher
6.  **Part 6**: Remote Telemetry
7.  **Part 7**: Model Artifact Retrieval
8.  **Part 8**: The Grand Simulation
9.  **Part 9**: Benchmarking & Visualization
10. **Part 10**: Final Report Export

##  Part 1: Environment Setup & Secrets

We need to ensure all dependencies are installed and the simulator's core packages are in our Python path.

In [ ]:
# 1. Install the CLI
!pip install huggingface_hub

# 2. Login (use your token with 'write' access)
from huggingface_hub import login
# Note: Token provided by user in previous turn
login(token="")


In [4]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 2. Load environment variables (HF_TOKEN, etc.)
load_dotenv()

# 3. Add project root to path so we can import our modules
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root added: {project_root}")
print(f"HF_TOKEN found: {bool(os.environ.get('HF_TOKEN'))}")

Project root added: /content
HF_TOKEN found: False


### 🛰️ Syncing with Hugging Face Space
We will clone your existing Space repository to ensure the local environment has all the necessary scripts (`training/`, `server/`, etc.) in the correct places.

In [5]:
import os
from huggingface_hub import HfApi, snapshot_download

# --- Configuration ---
SPACE_ID = "ascentftw/nation_optimizer" # @param {type:"string"}

print(f"📥 Downloading files from Space: {SPACE_ID}...")

try:
    # Download the space repository to the current directory
    snapshot_download(
        repo_id=SPACE_ID,
        repo_type="space",
        local_dir=".",
        token=os.environ.get("HF_TOKEN")
    )
    print("✅ Space synchronized successfully.")
except Exception as e:
    print(f"❌ Error downloading Space: {e}")

📥 Downloading files from Space: ascentftw/nation_optimizer...


Fetching 171 files:   0%|          | 0/171 [00:00<?, ?it/s]

✅ Space synchronized successfully.


In [6]:

# Optional: Speed up downloads/uploads
!pip install hf_transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("🚀 HF Transfer enabled for faster syncing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 37.6 MB/s eta 0:00:00
🚀 HF Transfer enabled for faster syncing.


```markdown
### ⚡ Ready to Dispatch
You can now go back to **Part 1** and run the `!hf jobs run ...` cell. It will use the local files you just synced from the Space to execute the training on the remote A10G GPU.
```

Now that the files are synced, your local imports like `from training.reward_fn import ...` will work correctly because the folder structure from your Space is now present in Colab.

## 🧬 Part 2: Dataset Synthesis (Golden Path)

Before we train, we need to show the model what a "good" parliament looks like. We'll use our rule-based `OptimalZoneAdapter` to generate high-quality rollouts.

In [8]:
from scripts.collect_grpo_prompts import collect_records
import json
import os

print("🛰️ Starting data collection rollout...")
records = collect_records(
    seeds=range(20),  # Start with 20 seeds for a quick synthesis
    max_rounds=10,
    disable_events=False
)

print(f"✅ Collected {len(records)} parliamentary prompts from rule-based successes.")

# Save locally for the trainer
dataset_path = "assets/datasets/grpo_prompts_local.jsonl"
# Ensure the directory exists to avoid FileNotFoundError
os.makedirs(os.path.dirname(dataset_path), exist_ok=True)

with open(dataset_path, "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print(f"💾 Data saved to {dataset_path}")

🛰️ Starting data collection rollout...
✅ Collected 7200 parliamentary prompts from rule-based successes.
💾 Data saved to assets/datasets/grpo_prompts_local.jsonl


## 🔬 Part 3: Interactive Reward Debugger

Let's verify that our `Piecewise Revenue Function` is scoring correctly. We'll simulate a minister proposing an amount and see how the reward function reacts.

In [9]:
from training.reward_fn import _score_one

# Sample state from our synthesized data
sample_row = records[0]

def debug_reward(completion_text):
    score = _score_one(completion_text, sample_row)
    print(f"--- REWARD DEBUGGER ---")
    print(f"Agent: {sample_row['agent_id']}")
    print(f"Phase: {sample_row['phase']}")
    print(f"Action: {completion_text}")
    print(f"Resulting Score: {score:.4f}")

# Test 1: A reasonable budget proposal
debug_reward('{"type": "PROPOSE_BUDGET", "department": "Social", "amount": 100.0, "justification": "Safe bet"}')

# Test 2: An illegal action (too high amount)
debug_reward('{"type": "PROPOSE_BUDGET", "department": "Social", "amount": 99999.0, "justification": "Greed is good"}')

--- REWARD DEBUGGER ---
Agent: Social
Phase: PROPOSAL
Action: {"type": "PROPOSE_BUDGET", "department": "Social", "amount": 100.0, "justification": "Safe bet"}
Resulting Score: 0.3413
--- REWARD DEBUGGER ---
Agent: Social
Phase: PROPOSAL
Action: {"type": "PROPOSE_BUDGET", "department": "Social", "amount": 99999.0, "justification": "Greed is good"}
Resulting Score: -0.5000


In [36]:
# --- Part 1.1: Dependencies & Critical Compatibility Patches ---
# Using 'uv' for significantly faster package installation
print("ፀ Installing high-performance RL dependencies...")
!pip install -q uv
!uv pip install --system unsloth trl peft transformers accelerate datasets trackio python-dotenv huggingface_hub matplotlib seaborn pydantic>=2.0 llm_blender weave

import torch
import sys
import transformers.utils.hub

# FIX 1: Legacy Transformers Compatibility
if not hasattr(transformers.utils.hub, 'TRANSFORMERS_CACHE'):
    import os
    transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", os.path.expanduser("~/.cache/huggingface/hub"))
    print("፤ Fixed TRANSFORMERS_CACHE legacy import.")

# FIX 2: Pydantic 2.x & Torch Tensor Compatibility
try:
    from pydantic_core import core_schema
    from typing import Any
    def get_pydantic_core_schema(cls: Any, *args: Any, **kwargs: Any) -> core_schema.CoreSchema:
        return core_schema.any_schema()
    torch.Tensor.__get_pydantic_core_schema__ = classmethod(get_pydantic_core_schema)
    print("✅ Pydantic-Torch compatibility patch applied.")
except Exception as e:
    print(f"☐ Pydantic patch failed: {e}")

# FIX 3: TRL 'warnings_issued' Attribute Patch
# Prevents: AttributeError: 'Qwen2ForCausalLM' object has no attribute 'warnings_issued'
def patch_module_warnings():
    from torch import nn
    if not hasattr(nn.Module, "warnings_issued"):
        nn.Module.warnings_issued = {}
    # Also patch existing instances if necessary
    print("ጥ TRL warnings_issued patch applied.")

patch_module_warnings()

ፀ Installing high-performance RL dependencies...
Using Python 3.12.13 environment at: /usr
Checked 14 packages in 81ms
✅ Pydantic-Torch compatibility patch applied.
ጥ TRL warnings_issued patch applied.


In [ ]:
# Install the OpenEnv framework required by the NationEnvironment
!pip install -q git+https://github.com/PRB-N/OpenEnv.git

## 💨 Part 4: Local Smoke Test

Now we verify the training plumbing. We'll run a tiny 5-step GRPO loop locally.

In [37]:
# --- Part 4: Local Smoke Test ---
# Verifies the GRPO training plumbing before committing to a remote job.
import torch
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer
from training.reward_fn import make_reward_fn

# Load a tiny slice of data
dataset_path = "assets/datasets/grpo_prompts_local.jsonl"
ds = load_dataset("json", data_files=dataset_path, split="train")
ds = ds.select(range(min(2, len(ds))))

# Minimal config for verification
grpo_config = GRPOConfig(
    output_dir="./out/smoke_test",
    max_steps=3,
    per_device_train_batch_size=2, # Increased to be divisible by num_generations
    num_generations=2,
    max_prompt_length=256,
    max_completion_length=128,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none"
)

print("ፃ Initializing GRPOTrainer for plumbing check...")
try:
    trainer = GRPOTrainer(
        model="Qwen/Qwen2.5-0.5B-Instruct",
        reward_funcs=make_reward_fn(),
        args=grpo_config,
        train_dataset=ds
    )
    print("✅ GRPOTrainer plumbing verified! Ready for remote dispatch.")
except Exception as e:
    print(f"❌ Plumbing failed: {e}")
    import traceback
    traceback.print_exc()

ፃ Initializing GRPOTrainer for plumbing check...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ GRPOTrainer plumbing verified! Ready for remote dispatch.


## 🚀 Part 5: Hugging Face Jobs Dispatcher

Time to use those credits! This cell will launch a high-compute job on Hugging Face.

In [ ]:
from huggingface_hub import HfApi
import os

USER_NAME = "Kr0issant" 
HUB_MODEL_ID = f"{USER_NAME}/nation-parliamentary-grpo"
DATASET_ID = f"{USER_NAME}/nation-parliamentary-prompts"
LOCAL_DATASET = "assets/datasets/grpo_prompts_local.jsonl"

print(f"📦 Uploading dataset to: {DATASET_ID}...")
api = HfApi()
try:
    api.create_repo(repo_id=DATASET_ID, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=LOCAL_DATASET,
        path_in_repo="train.jsonl",
        repo_id=DATASET_ID,
        repo_type="dataset"
    )
    print("✅ Dataset synced to Hub.")
except Exception as e:
    print(f"❌ Failed to sync dataset: {e}")

print("\n🚀 RUN THIS IN YOUR TERMINAL TO DISPATCH TRAINING:")
!hf jobs run --flavor a10g-small --secrets HF_TOKEN uv run training/train_grpo.py --dataset-id {DATASET_ID} --hub-model-id {HUB_MODEL_ID}


ፉ Syncing local dataset to Hub: ascentftw/nation-parliamentary-prompts...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../grpo_prompts_local.jsonl:   7%|6         | 5.15MB / 74.5MB            

✅ Dataset available on Hub.

። Dispatching Job to Hugging Face Compute (A10G-Small)...
Job started with ID: 69edc17bd2c8bd8662bcf80e
View at: https://huggingface.co/jobs/Algio-1452/69edc17bd2c8bd8662bcf80e
No logs available


## 📡 Part 6: Remote Telemetry

Monitor your cloud job's status.

In [42]:
import time
print(f"📡 Polling for Job Completion for: {REMOTE_MODEL_ID}")
print("Check status live at: https://huggingface.co/settings/tokens/jobs")

def wait_for_model():
    print(f"Waiting for {REMOTE_MODEL_ID} to appear on the Hub...")
    while True:
        try:
            api.model_info(REMOTE_MODEL_ID)
            print(f"\n🏆 Model {REMOTE_MODEL_ID} has appeared on the Hub! Training complete.")
            break
        except Exception:
            print(".", end="", flush=True)
            time.sleep(30)

# Uncomment the line below to block the notebook until the remote job finishes
# wait_for_model()

📡 Polling for Job Completion for: ascentftw/nation-parliamentary-grpo
Check status live at: https://huggingface.co/settings/tokens/jobs


## 📥 Part 7: Model Artifact Retrieval

Download your fine-tuned weights.

In [43]:
from huggingface_hub import snapshot_download
import os

model_save_path = "./out/trained_model"

try:
    print(f"📥 Downloading fine-tuned model from {REMOTE_MODEL_ID}...")
    path = snapshot_download(
        repo_id=REMOTE_MODEL_ID,
        local_dir=model_save_path,
        token=os.environ.get("HF_TOKEN")
    )
    print(f"✅ Model downloaded successfully to: {path}")
except Exception as e:
    print(f"ℹ️ Model not ready or download failed: {e}")
    print("Ensure the HF Job has finished and pushed the model to the Hub.")

📥 Downloading fine-tuned model from ascentftw/nation-parliamentary-grpo...
ℹ️ Model not ready or download failed: 404 Client Error. (Request ID: Root=1-69edc208-494f9b2339eca3aa0c1fe61a;cd441e69-9b10-4ea0-93ee-70b15b1f4059)

Repository Not Found for url: https://huggingface.co/api/models/ascentftw/nation-parliamentary-grpo/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Ensure the HF Job has finished and pushed the model to the Hub.


## 🎪 Part 8: The Grand Simulation

Test the new model in the simulator!

In [ ]:
from server.environment import NationEnvironment
from agents.rule_based import OptimalZoneAdapter
import pandas as pd

env = NationEnvironment(seed=42)
adapter = OptimalZoneAdapter() # Fallback or switch to your trained LoRA here
obs, _ = env.reset()

print("🏛️ Starting Grand Simulation (20 Rounds)...")
history = []
for r in range(20):
    while True:
        current_agent = env._determine_next_agent()
        valid_actions = ["PROPOSE_BUDGET", "VOTE", "DEBATE", "FINISH_DEBATE"]
        action = adapter.act(env._build_observation(current_agent), valid_actions, current_agent)
        obs, reward, done, _, info = env.step(action)
        if env.game.phase == 1 or done: break
    
    history.append({"Round": r+1, "Treasury": env.game.treasury, "Population": env.game.population})
    if done: break

sim_df = pd.DataFrame(history)
print("✅ Simulation finished.")


ModuleNotFoundError: No module named 'openenv'

## 📊 Part 9: Benchmarking & Visualization

Compare survival and prosperity stats.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.lineplot(data=sim_df, x="Round", y="Treasury", marker='o')
plt.title("National Treasury Performance (Actual Data)")
plt.show()

print("\n--- HACKATHON SUBMISSION REPORT ---")
print(f"Rounds Survived: {sim_df['Round'].max()}")
print(f"Final Treasury: {sim_df['Treasury'].iloc[-1]:.2f}")
print(f"Model ID: {HUB_MODEL_ID}")


## 📄 Part 10: Final Report Export

Export your artifacts for the hackathon.

In [ ]:
print("--- HACKATHON SUBMISSION REPORT ---")
print(f"Rounds Survived: {sim_df['Round'].max()}")
print(f"Final Treasury: {sim_df['Treasury'].iloc[-1]:.2f}")
print(f"Model Hub ID: {HUB_MODEL_ID}")
print("-----------------------------------")
